# Fakeddit Fake News Detector — Full Pipeline

This notebook trains a multimodal fake news detector using a CLIP backbone on the Fakeddit dataset.

**Before running:**
1. Set runtime to GPU: **Runtime → Change runtime type → T4 GPU**
2. Make sure your Google Drive has the following structure:
```
My Drive/fakeddit/
├── data/
│   ├── subset_train.tsv       (90,000 samples)
│   ├── subset_validate.tsv    (4,500 samples)
│   └── subset_test.tsv        (4,500 samples)
└── images/
    └── (downloaded images)
```
3. Run all cells top to bottom — **do not skip any**.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install Dependencies

In [2]:
!pip install -q torch transformers scikit-learn tqdm pandas numpy

## 3. Clone Repo

In [3]:
import os

REPO_URL = 'https://github.com/Girish4679/FakeNewsDetector.git'

if not os.path.exists('/content/FakeNewsDetector'):
    !git clone {REPO_URL} /content/FakeNewsDetector
else:
    print('Repo already cloned, pulling latest...')
    !git -C /content/FakeNewsDetector pull

%cd /content/FakeNewsDetector
print('Working directory:', os.getcwd())

Repo already cloned, pulling latest...
Already up to date.
/content/FakeNewsDetector
Working directory: /content/FakeNewsDetector


## 4. Download CLIP Weights

Downloads ~600MB of pretrained CLIP weights from HuggingFace. Only happens once per Colab session.

In [4]:
!python src/model.py

Loading CLIP (this downloads weights on first run ~600MB)...
Loading weights: 100% 398/398 [00:00<00:00, 807.43it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Logits: tensor([0.0835, 0.0854, 0.0822, 0.0822])
Probs:  tensor([0.5209, 0.5213, 0.5205, 0.5205])

Total params:     149,899,906
Trainable params: 279,169  (0.2%)

Expected: ~150M total, ~200K trainable (frozen CLIP, MLP only)
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.ra

## 5. Train the Model

Trains a frozen CLIP + fusion MLP on 90k samples. Checkpoints are saved to Drive after every epoch.

In [5]:
!python src/train.py \
  --data_dir "/content/drive/MyDrive/fakeddit/data" \
  --image_dir "/content/images" \
  --checkpoint_dir "/content/drive/MyDrive/fakeddit/checkpoints" \
  --subset \
  --epochs 5 \
  --batch_size 32

Device: cuda
GPU: NVIDIA A100-SXM4-40GB

Building dataloaders...
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
FakedditDataset: 90,000 samples  (/content/drive/MyDrive/fakeddit/data/subset_train.tsv)
  train: 90,000 samples → 2,812 batches of 32
HTTP Error 500 thrown while requesting HEAD https://huggingface.co/openai/clip-vit-base-patch16/resolve/main/chat_template.jinja
Retrying in 1s [Retry 1/5].
HTTP Error 500 thrown while requesting HEAD https://huggingface.co/openai/clip-vit-base-patch16/resolve/main/chat_template.jinja
Retrying in 2s [Retry 2/5].
HTTP Error 500 thrown while requesting HEAD https://huggingface.co/openai/clip-vit-base-patch16/resolve/main/chat_template.jinja
Retrying in 4s [Retry 3/5].
HTTP Error 500 th

In [9]:
path = '/content/FakeNewsDetector/src/evaluate.py'
with open(path, 'r') as f:
    content = f.read()

content = content.replace(
    'p.add_argument("--split",      default="test", choices=["train", "validate", "test"])',
    'p.add_argument("--split",      default="test", choices=["train", "validate", "test"])\n    p.add_argument("--subset", action="store_true", help="Use subset TSV")'
)
content = content.replace(
    'tsv_path = Path(args.data_dir) / SPLIT_FILES[args.split]',
    'subset_files = {"train": "subset_train.tsv", "validate": "subset_validate.tsv", "test": "subset_test.tsv"}\n    tsv_path = Path(args.data_dir) / (subset_files[args.split] if args.subset else SPLIT_FILES[args.split])'
)
content = content.replace('require_image=True', 'require_image=False')
content = content.replace(
    'ckpt = torch.load(args.checkpoint, map_location=device)',
    'ckpt = torch.load(args.checkpoint, map_location=device, weights_only=False)'
)
content = content.replace(
    'model.load_state_dict(ckpt["model"])',
    'state_dict = {k: v for k, v in ckpt["model"].items() if not k.startswith("loss_fn")}\n    model.load_state_dict(state_dict)'
)
with open(path, 'w') as f:
    f.write(content)
print('evaluate.py patched')

evaluate.py patched


## 6. Evaluate on Test Set

Loads the best checkpoint and runs it on the held-out test set. Prints F1, AUC, accuracy, and a confusion matrix.

In [10]:
!python src/evaluate.py \
  --checkpoint "/content/drive/MyDrive/fakeddit/checkpoints/checkpoint_best.pt" \
  --data_dir "/content/drive/MyDrive/fakeddit/data" \
  --image_dir "/content/images" \
  --subset

Device: cuda

Checkpoint from epoch 5
  Saved val metrics — F1: 0.8484  AUC: 0.9241

Loading weights: 100% 398/398 [00:00<00:00, 821.83it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
FakedditDataset: 4,500 samples  (/content/drive/MyDrive/fakeddit/data/subset_test.tsv)
/content/Fak